# Agregação espacial H3

Este notebook transforma a camada geocodificada em artefatos analíticos prontos para a API e a interface.

## Saídas principais

- `hex_segmento_atual.parquet`
- `areas_detalhe_atual.parquet`
- `qualidade_espacial_resumo.json`

In [ ]:
from pathlib import Path
import json
import re

try:
    import numpy as np
    import pandas as pd
    import pyarrow  # noqa: F401
except ImportError:
    !pip -q install numpy pandas pyarrow
    import numpy as np
    import pandas as pd
    import pyarrow  # noqa: F401

try:
    import h3
except ImportError:
    !pip -q install h3
    import h3

SNAPSHOT_MES = '2026-04'
CIDADE_ALVO = 'Praia Grande - SP'
RESOLUCAO_H3 = 9
LIMITE_SEGMENTOS = 12
SNAPSHOT = Path('/content/dados_brutos/cnpj') / SNAPSHOT_MES
PASTA_PREPARADO = SNAPSHOT / 'processado' / 'preparado'
PASTA_ANALITICO = SNAPSHOT / 'processado' / 'analitico'
ESTAB_GEO = PASTA_PREPARADO / 'estabelecimentos_geocodificados.parquet'
CNAES_RECORTE = PASTA_PREPARADO / 'cnaes_recorte.parquet'
MOTIVOS_RECORTE = PASTA_PREPARADO / 'motivos_situacao_recorte.parquet'

PASTA_ANALITICO.mkdir(parents=True, exist_ok=True)

if not ESTAB_GEO.exists():
    raise FileNotFoundError(f'Arquivo geocodificado ausente: {ESTAB_GEO}')

DATA_REFERENCIA = pd.Period(SNAPSHOT_MES, freq='M').end_time.normalize()
ESTAB_GEO

In [ ]:
estabelecimentos = pd.read_parquet(ESTAB_GEO)
cnaes_recorte = pd.read_parquet(CNAES_RECORTE) if CNAES_RECORTE.exists() else pd.DataFrame(columns=['cnae', 'descricao'])
motivos_recorte = pd.read_parquet(MOTIVOS_RECORTE) if MOTIVOS_RECORTE.exists() else pd.DataFrame(columns=['motivo', 'descricao'])

estabelecimentos.shape

In [ ]:
def slugificar(texto: str) -> str:
    texto = (texto or '').strip().lower()
    texto = re.sub(r'[^a-z0-9]+', '-', texto)
    return texto.strip('-') or 'segmento'

def meses_entre(data_inicial: pd.Timestamp, data_final: pd.Timestamp) -> float:
    if pd.isna(data_inicial):
        return np.nan
    return max((data_final - data_inicial).days / 30.4375, 0)

def confianca_dominante(valores: pd.Series) -> str:
    ordem = {'alta': 3, 'media': 2, 'baixa': 1}
    contagem = valores.fillna('baixa').value_counts().to_dict()
    if not contagem:
        return 'baixa'
    return sorted(contagem.items(), key=lambda item: (item[1], ordem.get(item[0], 0)), reverse=True)[0][0]

def nome_area_hex(serie_bairros: pd.Series, hex_id: str) -> str:
    bairros = serie_bairros.fillna('').astype(str).str.strip()
    bairros = bairros.loc[bairros != '']
    if bairros.empty:
        return f'Hex {hex_id[-6:]}'
    return bairros.value_counts().index[0].title()

def observacao_area(ativos: int, taxa_fechamento: float, confianca: str) -> str:
    if confianca == 'baixa':
        return 'Cobertura geográfica ainda fraca. Use esta leitura com cautela.'
    if taxa_fechamento >= 0.25:
        return 'Área com sinais recentes de volatilidade e fechamento acima do desejável.'
    if ativos >= 10:
        return 'Área com massa crítica relevante e leitura espacial já utilizável para o POC.'
    return 'Área em formação, com leitura espacial inicial útil para exploração.'


In [ ]:
geocodificados = estabelecimentos.loc[estabelecimentos['latitude'].notna() & estabelecimentos['longitude'].notna()].copy()
if geocodificados.empty:
    raise ValueError('Nenhum estabelecimento geocodificado com latitude/longitude disponível.')

geocodificados['hex_id'] = geocodificados.apply(lambda linha: h3.latlng_to_cell(float(linha['latitude']), float(linha['longitude']), RESOLUCAO_H3), axis=1)
geocodificados['idade_meses'] = geocodificados['data_inicio_atividade'].apply(lambda data: meses_entre(data, DATA_REFERENCIA))
geocodificados['abertura_24m'] = geocodificados['data_inicio_atividade'].ge(DATA_REFERENCIA - pd.DateOffset(months=24))
geocodificados['baixa_24m'] = geocodificados['flag_baixada'] & geocodificados['data_situacao_cadastral'].ge(DATA_REFERENCIA - pd.DateOffset(months=24))

segmentos_base = geocodificados.loc[geocodificados['flag_ativa']].groupby('cnae_fiscal_principal').size().sort_values(ascending=False).head(LIMITE_SEGMENTOS).reset_index(name='ativos_referencia')
if not cnaes_recorte.empty:
    segmentos_base = segmentos_base.merge(cnaes_recorte.rename(columns={'cnae': 'cnae_fiscal_principal', 'descricao': 'segmento_nome'}), on='cnae_fiscal_principal', how='left')
else:
    segmentos_base['segmento_nome'] = pd.NA
segmentos_base['segmento_nome'] = segmentos_base['segmento_nome'].fillna(segmentos_base['cnae_fiscal_principal'].apply(lambda codigo: f'CNAE {codigo}'))
segmentos_base['segmento_id'] = segmentos_base.apply(lambda linha: slugificar(f"{linha['segmento_nome']}-{linha['cnae_fiscal_principal']}"), axis=1)
geocodificados = geocodificados.merge(segmentos_base[['cnae_fiscal_principal', 'segmento_id', 'segmento_nome']], on='cnae_fiscal_principal', how='inner')

segmentos_base.head(5)

In [ ]:
centros_hex = geocodificados.groupby('hex_id').agg(
    nome_area=('bairro', lambda serie: nome_area_hex(serie, serie.name)),
    confianca_hex=('confianca_geografica', confianca_dominante),
).reset_index()
centros_hex[['lat_centro', 'lon_centro']] = centros_hex['hex_id'].apply(lambda hex_id: pd.Series(h3.cell_to_latlng(hex_id)))

lon_min, lon_max = centros_hex['lon_centro'].min(), centros_hex['lon_centro'].max()
lat_min, lat_max = centros_hex['lat_centro'].min(), centros_hex['lat_centro'].max()
centros_hex['centro_x'] = ((centros_hex['lon_centro'] - lon_min) / max(lon_max - lon_min, 1e-9) * 260 + 140).round().astype(int)
centros_hex['centro_y'] = (260 - ((centros_hex['lat_centro'] - lat_min) / max(lat_max - lat_min, 1e-9) * 120)).round().astype(int)

hexs_presentes = set(centros_hex['hex_id'])
centros_hex['hexagonos_vizinhos'] = centros_hex['hex_id'].apply(lambda hex_id: sorted([vizinho for vizinho in h3.grid_disk(hex_id, 1) if vizinho != hex_id and vizinho in hexs_presentes]))

centros_hex.head(3)

In [ ]:
analitico = geocodificados.groupby(['hex_id', 'segmento_id', 'segmento_nome', 'cnae_fiscal_principal']).agg(
    ativos=('flag_ativa', 'sum'),
    baixadas_atuais=('flag_baixada', 'sum'),
    aberturas_24m=('abertura_24m', 'sum'),
    baixas_24m=('baixa_24m', 'sum'),
    idade_mediana_meses=('idade_meses', 'median'),
    confianca_geografica=('confianca_geografica', confianca_dominante),
).reset_index()
complementares = geocodificados.loc[geocodificados['flag_ativa']].groupby('hex_id').size().rename('ativos_hex_total').reset_index()
analitico = analitico.merge(complementares, on='hex_id', how='left')
analitico['densidade_segmento'] = analitico['ativos'].astype(float)
analitico['densidade_complementares'] = (analitico['ativos_hex_total'] - analitico['ativos']).clip(lower=0).astype(float)
analitico['taxa_fechamento'] = (analitico['baixas_24m'] / (analitico['ativos'] + analitico['baixas_24m']).replace(0, np.nan)).fillna(0).round(4)
analitico['percentil_saturacao'] = analitico.groupby('segmento_id')['ativos'].rank(pct=True, method='average').mul(100).round().astype(int)
analitico = analitico.merge(centros_hex[['hex_id', 'nome_area', 'centro_x', 'centro_y', 'hexagonos_vizinhos']], on='hex_id', how='left')
analitico['hexagonos_vizinhos'] = analitico['hexagonos_vizinhos'].apply(json.dumps)

analitico.head(5)

In [ ]:
motivos_lookup = dict(zip(motivos_recorte['motivo'].astype(str), motivos_recorte['descricao'].astype(str))) if not motivos_recorte.empty else {}
ultimos_6_meses = pd.period_range(DATA_REFERENCIA - pd.DateOffset(months=5), DATA_REFERENCIA, freq='M')
areas_detalhe = []

for hex_id, grupo in geocodificados.groupby('hex_id'):
    nome_area = nome_area_hex(grupo['bairro'], hex_id)
    confianca = confianca_dominante(grupo['confianca_geografica'])
    ativos = int(grupo['flag_ativa'].sum())
    taxa_hex = float(grupo['baixa_24m'].sum() / max(grupo['flag_ativa'].sum() + grupo['baixa_24m'].sum(), 1))
    motivos = grupo.loc[grupo['flag_baixada']].assign(motivo_label=lambda df: df['motivo_situacao_cadastral'].fillna('').astype(str).str.zfill(2).map(motivos_lookup).fillna(df['motivo_situacao_cadastral'].fillna('Sem motivo informado'))).groupby('motivo_label').size().sort_values(ascending=False).head(5).reset_index(name='quantidade').rename(columns={'motivo_label': 'motivo'})

    serie_aberturas = []
    serie_baixas = []
    for periodo in ultimos_6_meses:
        inicio = periodo.to_timestamp(how='start')
        fim = periodo.to_timestamp(how='end')
        serie_aberturas.append(int(grupo['data_inicio_atividade'].between(inicio, fim, inclusive='both').sum()))
        serie_baixas.append(int((grupo['flag_baixada'] & grupo['data_situacao_cadastral'].between(inicio, fim, inclusive='both')).sum()))

    vizinhos = centros_hex.loc[centros_hex['hex_id'] == hex_id, 'hexagonos_vizinhos'].iloc[0] if hex_id in set(centros_hex['hex_id']) else []
    areas_detalhe.append({
        'hex_id': hex_id,
        'nome_area': nome_area,
        'motivos_baixa_json': json.dumps(motivos.to_dict(orient='records'), ensure_ascii=False),
        'serie_aberturas_json': json.dumps(serie_aberturas),
        'serie_baixas_json': json.dumps(serie_baixas),
        'observacao': observacao_area(ativos, taxa_hex, confianca),
        'hexagonos_vizinhos_json': json.dumps(vizinhos),
    })

areas_detalhe = pd.DataFrame(areas_detalhe)
areas_detalhe.head(3)

In [ ]:
analitico.to_parquet(PASTA_ANALITICO / 'hex_segmento_atual.parquet', index=False)
areas_detalhe.to_parquet(PASTA_ANALITICO / 'areas_detalhe_atual.parquet', index=False)

total_estabelecimentos = len(estabelecimentos)
com_coordenadas = int(estabelecimentos['latitude'].notna().sum())
qualidade_espacial = {
    'cidade': CIDADE_ALVO,
    'snapshot_referencia': SNAPSHOT_MES,
    'fonte_dados': 'real',
    'cobertura_geocodificacao': round(com_coordenadas / max(total_estabelecimentos, 1), 4),
    'confianca_alta': round(float((estabelecimentos['confianca_geografica'] == 'alta').sum()) / max(total_estabelecimentos, 1), 4),
    'confianca_media': round(float((estabelecimentos['confianca_geografica'] == 'media').sum()) / max(total_estabelecimentos, 1), 4),
    'confianca_baixa': round(float((estabelecimentos['confianca_geografica'] == 'baixa').sum()) / max(total_estabelecimentos, 1), 4),
    'enderecos_para_revisao': int(estabelecimentos['precisa_revisao_geocodificacao'].sum()),
    'arquivos_previstos': ['estabelecimentos_preparados.parquet', 'estabelecimentos_geocodificados.parquet', 'hex_segmento_atual.parquet', 'areas_detalhe_atual.parquet'],
    'arquivos_processados': ['estabelecimentos_geocodificados.parquet', 'hex_segmento_atual.parquet', 'areas_detalhe_atual.parquet'],
    'hexagonos_gerados': int(analitico['hex_id'].nunique()),
    'segmentos_gerados': int(analitico['segmento_id'].nunique())
}

with (PASTA_ANALITICO / 'qualidade_espacial_resumo.json').open('w', encoding='utf-8') as arquivo:
    json.dump(qualidade_espacial, arquivo, ensure_ascii=False, indent=2)

qualidade_espacial